In [1]:
from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../data/processed/merged_commentary.csv")
output_csv = Path("../data/processed/penalties.csv")

df = pd.read_csv(input_csv)


def extract_penalty(row):
    text = str(row["commentaryText"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })

    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"top (centre|center)", text):
        position, direction = 2, "top center"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"(centre|center) right", text):
        position, direction = 4, "middle right"
    elif re.search(
        r"(centre|center) of the goal|down the middle|middle of the goal",
        text
    ):
        position, direction = 5, "middle center"
    elif re.search(r"(centre|center) left", text):
        position, direction = 6, "middle left"

    elif re.search(r"bottom right", text):
        position, direction = 7, "bottom right"
    elif re.search(r"bottom (centre|center)", text):
        position, direction = 8, "bottom center"
    elif re.search(r"bottom left", text):
        position, direction = 9, "bottom left"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    elif re.search(r"to the right|misses to the right", text):
        position, direction = 4, "middle right"
    elif re.search(r"to the left|misses to the left", text):
        position, direction = 6, "middle left"
    else:
        position, direction = pd.NA, pd.NA

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
penalties.to_csv(output_csv, index=False)

print(penalties[
    [
        "eventId",
        "commentaryOrder",
        "commentaryText",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv.resolve())

          eventId  commentaryOrder  \
725        704285               42   
1795       704295               12   
3575       704310               51   
3889       704312              109   
4258       704316               29   
...           ...              ...   
526160     757932               44   
526731     757937               20   
527296  401862907               43   
527430  401862908               64   
527653  401862910               69   

                                           commentaryText  penalty_position  \
725     Goal! West Ham United 1, Aston Villa 1. Lucas ...                 9   
1795    Goal! Manchester City 1, Ipswich Town 1. Erlin...                 9   
3575    Penalty saved. Evanilson (Bournemouth) right f...                 7   
3889    Goal! Crystal Palace 2, Leicester City 2. Jean...                 5   
4258    Penalty saved. Cameron Archer (Southampton) ri...                 9   
...                                                   ...            

In [ ]:
from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../data/processed_others/merged_commentary.csv")
output_csv_others = Path("../data/processed_others/penalties.csv")

df = pd.read_csv(input_csv)


def extract_penalty(row):
    text = str(row["commentaryText"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })

    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"top (centre|center)", text):
        position, direction = 2, "top center"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"(centre|center) right", text):
        position, direction = 4, "middle right"
    elif re.search(
        r"(centre|center) of the goal|down the middle|middle of the goal",
        text
    ):
        position, direction = 5, "middle center"
    elif re.search(r"(centre|center) left", text):
        position, direction = 6, "middle left"

    elif re.search(r"bottom right", text):
        position, direction = 7, "bottom right"
    elif re.search(r"bottom (centre|center)", text):
        position, direction = 8, "bottom center"
    elif re.search(r"bottom left", text):
        position, direction = 9, "bottom left"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    elif re.search(r"to the right|misses to the right", text):
        position, direction = 4, "middle right"
    elif re.search(r"to the left|misses to the left", text):
        position, direction = 6, "middle left"
    else:
        position, direction = pd.NA, pd.NA

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
penalties.to_csv(output_csv_others, index=False)

print(penalties[
    [
        "eventId",
        "commentaryOrder",
        "commentaryText",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv_others.resolve())

In [2]:
penalties = pd.read_csv(output_csv)
penalties_saved = penalties["save"].sum()
penalties_goal = penalties["goal"].sum()
penalties_missed = penalties["miss"].sum()

print(f"saved: {penalties_saved}, goal: {penalties_goal}, missed: {penalties_missed}")
print(f"total: {len(penalties)}, goal in %: {penalties_goal/len(penalties)*100:.2f}%")

saved: 393, goal: 1659, missed: 39
total: 2091, goal in %: 79.34%


In [4]:
output_csv_others = Path("../data/processed_others/penalties.csv")

penalties_others = pd.read_csv(output_csv_others)
penalties_others_saved = penalties_others["save"].sum()
penalties_others_goal = penalties_others["goal"].sum()
penalties_others_missed = penalties_others["miss"].sum()

print(f"saved: {penalties_others_saved}, goal: {penalties_others_goal}, missed: {penalties_others_missed}")
print(f"total: {len(penalties_others)}, goal in %: {penalties_others_goal/len(penalties_others)*100:.2f}%")

saved: 1316, goal: 5527, missed: 115
total: 6958, goal in %: 79.43%
